# Hydranomic - Master Notebook WEGnology (Arquitetura Modular e Controlador)

Este notebook integra **todos os novos módulos analíticos do Hydranomic** em uma única interface interativa e executável em lote, projetada para a metodologia de execução de containers do **WEGnology Notebook** (baseada em Losant).

### Características desta versão:
1. **Execução Independente (Sem Ordem Obrigatória):** Cada bloco/célula possui resolução própria de dados e dependências. Você pode executar a célula de Machine Learning antes da de Gráficos, ou vice-versa, sem risco de erro `NameError` ou `KeyError`.
2. **Bloco Controlador Integrado:** Uma célula orquestradora central que aceita estados (`1: ML`, `2: Gráfico`, `3: Metas`, ou `'tudo'`) e lê automaticamente o `context.json` injetado por Workflows da WEGnology.
3. **Persistência Automática no `OUTPUT_DIR`:** Qualquer módulo executado salva seus artefatos (`grafico_anual.json`, `previsao_ml.json`, `config.json`, `progresso_meta.json`, `sumario_execucao.json`) na pasta monitorada pela aba **Outputs** da WEGnology para sincronização com **Application Files**.

--- 
## Bloco 1: Adaptador WEGnology e Configuração de Ambiente
*(Define as funções de detecção de `INPUT_DIR`, `OUTPUT_DIR` e leitura/escrita de JSON com fallbacks locais)*

In [1]:
import os
import sys
import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Garante que a pasta atual e a raiz do projeto estejam no sys.path
caminho_atual = Path.cwd()
if str(caminho_atual) not in sys.path:
    sys.path.insert(0, str(caminho_atual))

# --- ADAPTADOR WEGNOLOGY INLINE (Garante execução autônoma mesmo sem arquivos externos) ---
def get_input_dir() -> Path:
    """Retorna o diretório de entrada (INPUT_DIR no WEGnology ou workflows local)."""
    if "INPUT_DIR" in os.environ and os.path.isdir(os.environ["INPUT_DIR"]):
        return Path(os.environ["INPUT_DIR"])
    # Fallback procurando a pasta workflows local
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]:
        candidato = p / "workflows"
        if candidato.is_dir():
            return candidato
    return Path.cwd()

def get_output_dir() -> Path:
    """Retorna o diretório de saída (OUTPUT_DIR no WEGnology ou workflows local)."""
    if "OUTPUT_DIR" in os.environ:
        caminho = Path(os.environ["OUTPUT_DIR"])
        caminho.mkdir(parents=True, exist_ok=True)
        return caminho
    # Fallback local
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]:
        candidato = p / "workflows"
        if candidato.is_dir():
            return candidato
    saida = Path.cwd() / "outputs"
    saida.mkdir(parents=True, exist_ok=True)
    return saida

def ler_json(nome_arquivo: str, padrao: Any = None) -> Any:
    """Localiza e lê um JSON em INPUT_DIR ou caminhos alternativos."""
    dir_in = get_input_dir()
    caminho = dir_in / nome_arquivo
    if not caminho.exists():
        # Procura em caminhos relativos conhecidos
        for p in [Path.cwd() / "workflows" / nome_arquivo, Path.cwd() / nome_arquivo]:
            if p.exists():
                caminho = p
                break
    if not caminho.exists() or caminho.stat().st_size == 0:
        if padrao is not None:
            return padrao
        raise FileNotFoundError(f"Arquivo de entrada '{nome_arquivo}' não localizado em {dir_in}.")
    with open(caminho, "r", encoding="utf-8") as f:
        return json.load(f)

def gravar_json(nome_arquivo: str, dados: Any) -> Path:
    """Grava o JSON no OUTPUT_DIR da WEGnology com garantia de diretório criado."""
    dir_out = get_output_dir()
    caminho = dir_out / nome_arquivo
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False, indent=2)
    print(f"[WEGnology Storage] Arquivo salvo: {caminho}")
    return caminho

def ler_contexto_wegnology() -> Dict[str, Any]:
    """Lê context.json caso injetado por Workflow da WEGnology."""
    try:
        return ler_json("context.json", padrao={})
    except Exception:
        return {}

print(f"✓ Ambiente pronto!")
print(f"  - INPUT_DIR:  {get_input_dir()}")
print(f"  - OUTPUT_DIR: {get_output_dir()}")

ModuleNotFoundError: No module named 'matplotlib'

--- 
## Bloco 2: Parâmetros Injetáveis (Papermill / WEGnology Workflow)
*(Esta célula pode ser sobrescrita automaticamente durante a execução agendada no WEGnology)*

In [ ]:
# Parâmetros padrão de execução
ESTADO_DESEJADO = "tudo"        # Opções: 1 (ML), 2 (Grafico), 3 (Metas), 'tudo' (Executa todos)
ANO_REFERENCIA_GRAFICO = None    # Se None, calcula automaticamente o ano mais recente
TARIFA_AGUA_M3 = 59.6            # Tarifa de água em R$ por m³
N_ESTIMATORS_ML = 250           # Quantidade de árvores no RandomForest
ARQUIVO_HISTORICO = "conta_povoada.json" # Fonte de dados principal
PERSISTIR_ARQUIVOS = True       # Grava os JSONs no OUTPUT_DIR

--- 
## Bloco 3: Módulo de Gráfico Anual de Consumo (`data_chart`)
*(Pode ser executado a qualquer momento de forma independente)*

In [3]:
from datetime import datetime

def modulo_grafico_anual(
    dados_consumo: Optional[List[Dict[str, Any]]] = None,
    ano_referencia: Optional[int] = None,
    salvar: bool = True,
    exibir_grafico: bool = True
) -> Dict[str, Any]:
    """
    Processa os registros mensais e formata a série de 12 meses para o dashboard anual.
    Funciona de forma 100% autônoma carregando os dados caso não sejam passados.
    """
    # Carrega dados caso não tenham sido passados
    if dados_consumo is None:
        try:
            dados_consumo = ler_json(ARQUIVO_HISTORICO)
        except Exception:
            dados_consumo = ler_json("conta.json", padrao=[])

    # Identifica o ano se não fornecido
    if ano_referencia is None:
        anos = []
        for r in dados_consumo:
            partes = str(r.get("id", "")).split("-")
            if len(partes) == 2 and partes[0].isdigit():
                anos.append(int(partes[0]))
        ano_referencia = max(anos) if anos else datetime.now().year

    consumo_mensal = {}
    mes_atual = None

    for reg in dados_consumo:
        partes = str(reg.get("id", "")).split("-")
        if len(partes) == 2:
            try:
                a = int(partes[0])
                m = int(partes[1])
                if a == ano_referencia and 1 <= m <= 12:
                    consumo_mensal[m] = reg.get("consumo_m3")
                    mes_atual = m
            except ValueError:
                continue

    tags_meses = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
    resultado_grafico = {
        "ano_referencia": ano_referencia,
        "mes_atual": mes_atual,
        "unidade_medida": "m³",
        "meses": [
            {
                "mes": i,
                "rotulo": tags_meses[i - 1],
                "consumo_m3": consumo_mensal.get(i)
            }
            for i in range(1, 13)
        ]
    }

    if salvar:
        gravar_json("grafico_anual.json", resultado_grafico)

    if exibir_grafico:
        labels = [m["rotulo"] for m in resultado_grafico["meses"]]
        vals = [m["consumo_m3"] if m["consumo_m3"] is not None else 0 for m in resultado_grafico["meses"]]
        cores = ["#0284c7" if m["consumo_m3"] is not None else "#e2e8f0" for m in resultado_grafico["meses"]]

        plt.figure(figsize=(9, 3.5))
        plt.bar(labels, vals, color=cores)
        plt.title(f"Consumo Mensal de Água - Ano {ano_referencia} (m³)")
        plt.xlabel("Mês")
        plt.ylabel("Consumo (m³)")
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

    return resultado_grafico

# Teste autônomo da célula (funciona em qualquer ordem)
res_grafico = modulo_grafico_anual(ano_referencia=ANO_REFERENCIA_GRAFICO, salvar=PERSISTIR_ARQUIVOS)
print(f"✓ Gráfico processado com sucesso para o ano {res_grafico['ano_referencia']}!")

NameError: name 'ANO_REFERENCIA_GRAFICO' is not defined

--- 
## Bloco 4: Módulo de Machine Learning (`machinelearning`)
*(RandomForestRegressor com engenharia de atributos temporais e defasagem)*
*(Pode ser executado a qualquer momento de forma independente)*

In [ ]:
from sklearn.ensemble import RandomForestRegressor

def modulo_machine_learning(
    dados_consumo: Optional[List[Dict[str, Any]]] = None,
    tarifa_m3: float = TARIFA_AGUA_M3,
    n_estimators: int = N_ESTIMATORS_ML,
    salvar: bool = True
) -> Dict[str, Any]:
    """
    Treina o modelo preditivo com os registros históricos e gera a previsão de consumo e fatura.
    Funciona de forma 100% autônoma carregando a base se necessário.
    """
    if dados_consumo is None:
        try:
            dados_consumo = ler_json(ARQUIVO_HISTORICO)
        except Exception:
            dados_consumo = ler_json("conta.json", padrao=[])

    if not dados_consumo:
        raise ValueError("Não há dados suficientes para treinar o modelo de Machine Learning.")

    # Converte e organiza o DataFrame
    df = pd.DataFrame(dados_consumo)
    df = df.sort_values(by=["unidade_consumidora", "id"]).reset_index(drop=True)

    # Extração de variáveis temporais
    df["ano"] = df["id"].astype(str).str.split("-").str[0].astype(int)
    df["mes"] = df["id"].astype(str).str.split("-").str[1].astype(int)

    ano_inicial = df["ano"].min()
    mes_inicial = df.loc[df["ano"] == ano_inicial, "mes"].min()
    df["tempo_iniciofinal"] = (df["ano"] - ano_inicial) * 12 + (df["mes"] - mes_inicial)

    # Memória de consumo (Lag 1 do mês anterior)
    df["memoria_consumo"] = df.groupby("unidade_consumidora")["consumo_m3"].shift(1)
    df_treino = df.dropna(subset=["memoria_consumo"]).copy()

    features_x = ["mes", "tempo_iniciofinal", "memoria_consumo"]
    alvo_y = "consumo_m3"

    # Treinamento do modelo
    modelo = RandomForestRegressor(n_estimators=n_estimators, random_state=45)
    modelo.fit(df_treino[features_x], df_treino[alvo_y])

    # Projeção para o próximo ciclo
    ultima_linha = df.iloc[-1]
    ultimo_ano = int(ultima_linha["ano"])
    ultimo_mes = int(ultima_linha["mes"])
    ultimo_consumo = float(ultima_linha["consumo_m3"])
    ultimo_passo = int(ultima_linha["tempo_iniciofinal"])

    if ultimo_mes == 12:
        prox_mes = 1
        prox_ano = ultimo_ano + 1
    else:
        prox_mes = ultimo_mes + 1
        prox_ano = ultimo_ano

    prox_passo = ultimo_passo + 1
    dados_pred = pd.DataFrame([[prox_mes, prox_passo, ultimo_consumo]], columns=features_x)
    previsao_m3 = float(modelo.predict(dados_pred)[0])
    valor_fatura = round(previsao_m3 * tarifa_m3, 2)

    resultado_ml = {
        "ano": prox_ano,
        "mes": prox_mes,
        "unidade_consumidora": int(ultima_linha["unidade_consumidora"]),
        "ultimo_consumo_registrado_m3": ultimo_consumo,
        "previsao_m3": round(previsao_m3, 2),
        "tarifa_m3": tarifa_m3,
        "valor_estimado_conta": valor_fatura,
        "modelo": "RandomForestRegressor"
    }

    if salvar:
        gravar_json("previsao_ml.json", resultado_ml)

    return resultado_ml

# Teste autônomo da célula (funciona em qualquer ordem)
res_ml = modulo_machine_learning(tarifa_m3=TARIFA_AGUA_M3, salvar=PERSISTIR_ARQUIVOS)
print(f"✓ Previsão ML calculada: {res_ml['mes']:02d}/{res_ml['ano']} -> {res_ml['previsao_m3']} m³ (R$ {res_ml['valor_estimado_conta']:.2f})")

--- 
## Bloco 5: Módulo de Metas de Consumo (`metas`)
*(Gerenciamento de metas ativas e avaliação de status de estouro/alerta 80%)*
*(Pode ser executado a qualquer momento de forma independente)*

In [ ]:
def modulo_metas(
    consumo_medido_m3: Optional[float] = None,
    salvar: bool = True
) -> Dict[str, Any]:
    """
    Lê config.json e calcula o progresso da meta ativa.
    Se consumo_medido_m3 não for passado, obtém o último valor dos dados históricos.
    """
    # Carrega a configuração de metas
    config = ler_json("config.json", padrao={"meta_ativa": None, "solicitacao_exclusao": None})

    # Se o consumo não for informado, busca o mais recente
    if consumo_medido_m3 is None:
        try:
            dados = ler_json(ARQUIVO_HISTORICO)
            consumo_medido_m3 = float(dados[-1].get("consumo_m3", 0.0))
        except Exception:
            consumo_medido_m3 = 0.0

    meta = config.get("meta_ativa")
    if not meta:
        relatorio = {
            "meta_ativa": False,
            "mensagem": "Nenhuma meta de consumo ativa configurada no momento.",
            "consumo_atual_m3": consumo_medido_m3
        }
    else:
        limite = float(meta.get("valor", 1.0))
        atingimento = round((consumo_medido_m3 / limite) * 100, 2)
        saldo = round(limite - consumo_medido_m3, 2)
        if atingimento >= 100:
            status = "ESTOURADA"
        elif atingimento >= 80:
            status = "ALERTA_80"
        else:
            status = "NORMAL"

        relatorio = {
            "meta_ativa": True,
            "meta_nome": meta.get("nome", "Meta de Consumo"),
            "limite_m3": limite,
            "consumo_atual_m3": consumo_medido_m3,
            "percentual_atingido": atingimento,
            "saldo_restante_m3": saldo,
            "situacao": status
        }

    if salvar:
        gravar_json("progresso_meta.json", relatorio)

    return relatorio

# Teste autônomo da célula (funciona em qualquer ordem)
res_meta = modulo_metas(salvar=PERSISTIR_ARQUIVOS)
print("✓ Relatório de metas processado com sucesso!")
print(res_meta)

--- 
## Bloco 6: BLOCO CONTROLADOR PRINCIPAL (Orquestrador do Pipeline)
**Este é o bloco controlador central.** Ele pode:
- Ser acionado manualmente escolhendo o estado (`1`, `2`, `3` ou `'tudo'`).
- Ser acionado automaticamente por um **Workflow da WEGnology** que injeta `context.json` com `{"estado": 1}` ou `{"acao": "ml"}`.
- Executar cada módulo de forma coordenada e gravar o `sumario_execucao.json` no `OUTPUT_DIR` para consumo direto pelos dashboards.

In [ ]:
def bloco_controlador(estado_solicitado: Any = None) -> Dict[str, Any]:
    """
    Controlador mestre do pipeline analítico do Hydranomic.
    
    Estados suportados:
      - 1 ou 'ml': Executa apenas a previsão de Machine Learning.
      - 2 ou 'grafico': Executa apenas o processamento do gráfico anual.
      - 3 ou 'metas': Executa apenas a avaliação de metas de consumo.
      - 'tudo' ou 0: Executa o pipeline completo (Gráfico + ML + Metas).
    """
    print("=" * 60)
    print("       INICIANDO BLOCO CONTROLADOR HYDRANOMIC WEGNOLOGY       ")
    print("=" * 60)

    # 1. Verifica se há contexto recebido da WEGnology
    contexto = ler_contexto_wegnology()
    if contexto:
        print(f"[Contexto WEGnology Detectado]: {contexto}")

    # Prioridade do estado: argumento da função > contexto WEGnology > variável global ESTADO_DESEJADO
    estado = estado_solicitado or contexto.get("estado") or contexto.get("acao") or ESTADO_DESEJADO

    resultado_execucao = {
        "estado_solicitado": estado,
        "timestamp": datetime.now().isoformat(),
        "outputs_gerados": []
    }

    # Despachador de Ações (Match/Case resiliente)
    match str(estado).lower().strip():
        case "1" | "ml":
            print("▶ Executando Estado 1: Predição de Consumo via Machine Learning...")
            ml_res = modulo_machine_learning(salvar=True)
            resultado_execucao["ml"] = ml_res
            resultado_execucao["outputs_gerados"].append("previsao_ml.json")
            print(f"  -> Previsão calculada: {ml_res['previsao_m3']} m³ | R$ {ml_res['valor_estimado_conta']:.2f}")

        case "2" | "grafico":
            print("▶ Executando Estado 2: Geração de Dados para Gráfico Anual...")
            ano_param = contexto.get("ano_referencia", ANO_REFERENCIA_GRAFICO)
            chart_res = modulo_grafico_anual(ano_referencia=ano_param, salvar=True, exibir_grafico=False)
            resultado_execucao["grafico"] = chart_res
            resultado_execucao["outputs_gerados"].append("grafico_anual.json")
            print(f"  -> Gráfico consolidado para o ano {chart_res['ano_referencia']}.")

        case "3" | "metas":
            print("▶ Executando Estado 3: Avaliação de Metas de Consumo...")
            consumo_ctx = contexto.get("consumo_atual")
            meta_res = modulo_metas(consumo_medido_m3=consumo_ctx, salvar=True)
            resultado_execucao["metas"] = meta_res
            resultado_execucao["outputs_gerados"].append("progresso_meta.json")
            print(f"  -> Relatório de meta gerado: {meta_res}")

        case _:
            print("▶ Executando Pipeline Completo (Consolidação de Todos os Módulos)...")
            
            # 1. Gráfico Anual
            chart_res = modulo_grafico_anual(salvar=True, exibir_grafico=False)
            resultado_execucao["grafico"] = chart_res
            resultado_execucao["outputs_gerados"].append("grafico_anual.json")

            # 2. Machine Learning
            ml_res = modulo_machine_learning(salvar=True)
            resultado_execucao["ml"] = ml_res
            resultado_execucao["outputs_gerados"].append("previsao_ml.json")

            # 3. Metas de Consumo
            consumo_mais_recente = ml_res.get("ultimo_consumo_registrado_m3", 0.0)
            meta_res = modulo_metas(consumo_medido_m3=consumo_mais_recente, salvar=True)
            resultado_execucao["metas"] = meta_res
            resultado_execucao["outputs_gerados"].append("progresso_meta.json")

    # Grava o sumário da execução no OUTPUT_DIR para o WEGnology
    gravar_json("sumario_execucao.json", resultado_execucao)
    resultado_execucao["outputs_gerados"].append("sumario_execucao.json")

    print("=" * 60)
    print("       EXECUÇÃO DO CONTROLADOR CONCLUÍDA COM SUCESSO!       ")
    print(f"       Arquivos gerados no OUTPUT_DIR: {len(resultado_execucao['outputs_gerados'])}")
    print("=" * 60)
    return resultado_execucao

# Executa o controlador (Você pode passar 1, 2, 3 ou 'tudo')
res_controlador = bloco_controlador(ESTADO_DESEJADO)

--- 
## Bloco 7: Verificação e Auditoria de Arquivos no `OUTPUT_DIR`
*(Lista todos os arquivos persistidos que estão prontos para sincronização com os **Application Files** da WEGnology)*

In [ ]:
dir_saida = get_output_dir()
print(f"Status dos Arquivos em: {dir_saida}")
print("-" * 55)
arquivos_saida = list(dir_saida.glob("*.json"))
if not arquivos_saida:
    print("Nenhum arquivo JSON localizado na pasta de saída.")
else:
    for arq in arquivos_saida:
        tamanho = arq.stat().st_size
        print(f"  [✓ Pronto para WEGnology] {arq.name:<25} ({tamanho:>6} bytes)")
print("-" * 55)